# Affiner Wav2Vec2-BERT pour la reconnaissance vocale du mooré

Dans ce tutoriel, vous entraînez un modèle de reconnaissance automatique de la parole (ASR) qui transcrit
de l'audio en mooré, à partir du modèle pré-entraîné [`facebook/w2v-bert-2.0`](https://huggingface.co/facebook/w2v-bert-2.0).
Vous publiez ensuite le modèle obtenu sur le Hugging Face Hub.

*Ce tutoriel s'inspire de l'article « [Fine-Tune W2V2-Bert for low-resource ASR with 🤗 Transformers](https://huggingface.co/blog/fine-tune-w2v2-bert) ».*

## Vue d'ensemble

[Wav2Vec2-BERT](https://huggingface.co/docs/transformers/model_doc/wav2vec2-bert) est un modèle audio de
580 millions de paramètres, pré-entraîné sur 4,5 millions d'heures d'audio non transcrit dans plus de
143 langues. Il a appris à représenter la parole, mais pas encore à l'écrire : vous lui ajoutez une couche
de sortie qui prédit des **caractères**, et vous l'entraînez avec la fonction de perte **CTC**
(*Connectionist Temporal Classification*).

Contrairement à Whisper, qui génère sa transcription mot à mot, un modèle CTC transcrit tout l'audio en une
seule passe. Il est donc plus rapide, et il donne de bons résultats avec peu de données : ici, environ
5 h 40 d'audio en mooré.

Vous allez :

1. charger la configuration `moore` du corpus [`burkimbia/faso-speech`](https://huggingface.co/datasets/burkimbia/faso-speech)
   (5 171 extraits d'entraînement et 575 de validation) ;
2. normaliser les transcriptions et construire un vocabulaire de caractères ;
3. créer le tokeniseur, l'extracteur de caractéristiques audio et le processeur qui les regroupe ;
4. préparer l'audio à 16 kHz ;
5. entraîner le modèle pendant 10 époques en mesurant le taux d'erreur sur les mots
   ([WER](https://huggingface.co/spaces/evaluate-metric/wer)), et suivre les courbes avec
   [Trackio](https://huggingface.co/docs/trackio) ;
6. transcrire un extrait de validation, puis publier le modèle sur le Hub.

## Prérequis

**Logiciels**

- Le dépôt `first-experiments` cloné, avec l'environnement installé par `uv sync`, et ce notebook ouvert avec le noyau `.venv` du dépôt.
  Sur Google Colab, la première cellule installe exactement les versions figées dans le fichier `uv.lock` du
  dépôt (environ 5 minutes), puis redémarre la session : relancez alors cette même cellule et continuez.
- Un compte [Hugging Face](https://huggingface.co/join) et un [token d'accès](https://huggingface.co/settings/tokens) avec les droits **write** :
  - en local, placez-le dans un fichier `.env` à la racine du dépôt : `HF_TOKEN=hf_...` ;
  - sur Colab, ajoutez-le dans les secrets (icône 🔑) sous le nom `HF_TOKEN`.
- FFmpeg, utilisé par `torchcodec` pour décoder l'audio. Il est déjà installé sur Colab ; sur Ubuntu, installez-le avec `sudo apt install ffmpeg`.
- Un accès au jeu de données [`burkimbia/faso-speech`](https://huggingface.co/datasets/burkimbia/faso-speech). Il est publié sous
  licence CC BY-NC 4.0 (usage non commercial) et l'accès est validé manuellement : ouvrez sa page, remplissez
  le formulaire de demande, puis attendez l'approbation avant de lancer ce notebook.

**Tutoriels préalables**

Aucun. Une connaissance de base de Python et de PyTorch suffit.

## Ressources nécessaires

| Ressource | Besoin |
| --- | --- |
| GPU | **Obligatoire** : GPU NVIDIA avec au moins **16 Go** de mémoire (par exemple un T4 sur Colab). Un GPU de portable de 4 à 8 Go ne suffit pas pour l'entraînement. |
| Mémoire vive | 12 Go |
| Disque | Environ 20 Go (modèle de 2,3 Go, audio préparé, et deux points de sauvegarde de 7 Go chacun) |
| Réseau | Téléchargement d'environ 3 Go (modèle et corpus) |

## Durée

- **Lecture et exécution des cellules** : environ 30 minutes.
- **Calcul** : environ 15 minutes pour préparer l'audio, puis environ 2 h à 3 h d'entraînement sur un T4
  (environ 45 minutes sur un A100). Pour un premier essai plus court, réduisez `num_train_epochs`.

## 1. Préparer l'environnement

Cette cellule charge votre token Hugging Face, en local comme sur Colab.

In [ ]:
import os
import subprocess

if "COLAB_RELEASE_TAG" in os.environ:
    # Google Colab : installez exactement les versions figées dans uv.lock, puis redémarrez la session.
    if not os.path.exists("/content/.env_ready"):
        repo_dir = "/content/first-experiments"
        subprocess.run(["pip", "install", "-q", "uv"], check=True)
        if not os.path.exists(repo_dir):
            subprocess.run(
                ["git", "clone", "-q", "--depth", "1", "https://github.com/BurkimbIA/first-experiments.git", repo_dir],
                check=True,
            )
        subprocess.run(
            ["uv", "export", "--frozen", "--no-dev", "--no-hashes", "--no-emit-project", "-o", "/content/requirements.txt"],
            cwd=repo_dir,
            check=True,
        )
        subprocess.run(
            [
                "uv", "pip", "install", "--system", "-q", "-r", "/content/requirements.txt",
                "--extra-index-url", "https://download.pytorch.org/whl/cu126",
                "--index-strategy", "unsafe-best-match",
            ],
            check=True,
        )
        open("/content/.env_ready", "w").close()
        print("Installation terminée. La session redémarre : relancez cette cellule.")
        os.kill(os.getpid(), 9)

    from google.colab import userdata

    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
else:
    # Machine locale : le token est lu depuis le fichier .env à la racine du dépôt.
    from dotenv import load_dotenv

    load_dotenv()

Vérifiez que vous êtes bien connecté au Hub et qu'un GPU est disponible. Le nom de compte affiché sert plus
loin à nommer le dépôt du modèle.

In [ ]:
import torch
from huggingface_hub import whoami

hf_user = whoami()["name"]
print("Compte Hugging Face :", hf_user)
print("GPU :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "aucun GPU détecté")

## 2. Charger le corpus

Chargez la configuration `moore` de Faso Speech. Gardez uniquement l'audio et la transcription (`text`).

In [ ]:
from datasets import load_dataset

dataset = load_dataset("burkimbia/faso-speech", "moore")
dataset = dataset.select_columns(["audio", "text"])
dataset

Affichez quelques transcriptions au hasard pour vous faire une idée du texte.

In [ ]:
dataset["train"].select_columns(["text"]).shuffle(seed=0).select(range(10)).to_pandas()

## 3. Normaliser les transcriptions

Sans modèle de langue, il est très difficile de prédire la ponctuation à partir de l'audio : un point ou
une virgule ne correspond à aucun son. Supprimez-la, et mettez tout le texte en minuscules, pour que le
modèle n'ait pas à distinguer `a` de `A`. Seuls les caractères qui se prononcent sont conservés, y compris
les lettres propres au mooré (`ɛ`, `ɩ`, `ʋ`) et les voyelles nasales (`ã`, `ẽ`, `ĩ`, `õ`, `ũ`).

In [ ]:
import re

CHARS_TO_REMOVE = r"[\\\?\.\!\-\;\:\"%’'�»«\+”“\(\)‘\*,]"


def normalize_text(batch):
    text = re.sub(CHARS_TO_REMOVE, "", batch["text"].lower())
    batch["text"] = re.sub(r"\s+", " ", text).strip()
    return batch


dataset = dataset.map(normalize_text, desc="Normalisation")
dataset = dataset.filter(lambda text: len(text) > 0, input_columns="text")
dataset["train"].select_columns(["text"]).shuffle(seed=0).select(range(10)).to_pandas()

## 4. Construire le vocabulaire

Un modèle CTC classe chaque fragment d'audio dans un caractère. Le vocabulaire est donc l'ensemble des
caractères présents dans les transcriptions d'entraînement et de validation.

In [ ]:
chars = set()
for split in dataset.values():
    for text in split["text"]:
        chars.update(text)

vocab_dict = {char: index for index, char in enumerate(sorted(chars))}
print(f"{len(vocab_dict)} caractères :", "".join(sorted(chars)))

Complétez le vocabulaire avec trois jetons spéciaux :

- l'espace `" "` est remplacé par `|`, plus visible, qui sépare les mots ;
- `[UNK]` représente un caractère inconnu, absent du vocabulaire ;
- `[PAD]` sert de remplissage et de jeton *blank* de CTC. Ce jeton permet au modèle de séparer deux lettres
  identiques consécutives : pour écrire `"hello"`, le modèle prédit par exemple `h e l [PAD] l o`. Voir la
  section *Alignment* de l'article [Sequence Modeling with CTC](https://distill.pub/2017/ctc/).

In [ ]:
vocab_dict["|"] = vocab_dict.pop(" ")
vocab_dict["[UNK]"] = len(vocab_dict)
vocab_dict["[PAD]"] = len(vocab_dict)
print("Taille du vocabulaire :", len(vocab_dict))

## 5. Créer le tokeniseur, l'extracteur de caractéristiques et le processeur

Enregistrez le vocabulaire dans `outputs/w2v-bert-2.0-moore/vocab.json`, puis créez un
`Wav2Vec2CTCTokenizer` à partir de ce fichier.

In [ ]:
import json
import os

from transformers import Wav2Vec2CTCTokenizer

repo_name = "w2v-bert-2.0-moore"
output_dir = f"outputs/{repo_name}"
os.makedirs(output_dir, exist_ok=True)

vocab_path = os.path.join(output_dir, "vocab.json")
with open(vocab_path, "w", encoding="utf-8") as vocab_file:
    json.dump(vocab_dict, vocab_file, ensure_ascii=False)

tokenizer = Wav2Vec2CTCTokenizer(
    vocab_path,
    unk_token="[UNK]",
    pad_token="[PAD]",
    word_delimiter_token="|",
    bos_token=None,
    eos_token=None,
)

L'extracteur de caractéristiques transforme le signal audio brut en spectrogramme log-mel, c'est-à-dire
l'énergie de chaque bande de fréquence au cours du temps. Pour en savoir plus, voir
[ce chapitre du cours audio de Hugging Face](https://huggingface.co/learn/audio-course/chapter1/audio_data#the-frequency-spectrum).
L'extracteur n'apprend rien des données : chargez directement celui du modèle pré-entraîné.

Le `Wav2Vec2BertProcessor` regroupe ensuite l'extracteur et le tokeniseur dans un seul objet.

In [ ]:
from transformers import SeamlessM4TFeatureExtractor, Wav2Vec2BertProcessor

feature_extractor = SeamlessM4TFeatureExtractor.from_pretrained("facebook/w2v-bert-2.0")
processor = Wav2Vec2BertProcessor(feature_extractor=feature_extractor, tokenizer=tokenizer)

## 6. Préparer l'audio

Wav2Vec2-BERT a été pré-entraîné sur de l'audio échantillonné à **16 kHz**. Vos données doivent avoir la
même fréquence d'échantillonnage. Convertissez la colonne `audio` : `datasets` décode et rééchantillonne
alors chaque extrait à la lecture.

In [ ]:
from datasets import Audio

dataset = dataset.cast_column("audio", Audio(sampling_rate=16_000))

sample = dataset["train"][0]
print("Transcription :", sample["text"])
print("Forme du signal :", sample["audio"]["array"].shape)
print("Fréquence d'échantillonnage :", sample["audio"]["sampling_rate"])

Écoutez un extrait au hasard pour vérifier que l'audio correspond bien à sa transcription. Relancez la
cellule pour écouter un autre extrait.

In [ ]:
import random

import IPython.display as ipd

sample = dataset["train"][random.randint(0, len(dataset["train"]) - 1)]
print(sample["text"])
ipd.Audio(data=sample["audio"]["array"], rate=sample["audio"]["sampling_rate"])

Transformez maintenant chaque extrait au format attendu par le modèle :

- `input_features` : le spectrogramme calculé par l'extracteur ;
- `input_length` : le nombre de trames, utilisé pour former des lots d'extraits de longueurs proches ;
- `labels` : la transcription encodée par le tokeniseur.

Gardez d'abord une copie du jeu de validation avec son audio, pour tester le modèle à la fin.
Cette étape prend environ 15 minutes.

In [ ]:
raw_validation = dataset["validation"]


def prepare_dataset(batch):
    audio = batch["audio"]
    batch["input_features"] = processor(audio["array"], sampling_rate=audio["sampling_rate"]).input_features[0]
    batch["input_length"] = len(batch["input_features"])
    batch["labels"] = processor(text=batch["text"]).input_ids
    return batch


dataset = dataset.map(
    prepare_dataset,
    remove_columns=dataset["train"].column_names,
    num_proc=4,
    desc="Préparation de l'audio",
)
dataset

## 7. Définir le regroupement des lots

En parole, l'entrée (l'audio) est beaucoup plus longue que la sortie (le texte), et les deux n'ont pas le
même format. Ce collecteur complète donc séparément les `input_features` et les `labels` jusqu'à l'élément
le plus long de chaque lot. Les positions de remplissage des `labels` sont remplacées par `-100` pour être
ignorées dans le calcul de la perte.

In [ ]:
from dataclasses import dataclass

import torch


@dataclass
class DataCollatorCTCWithPadding:
    processor: Wav2Vec2BertProcessor
    padding: bool | str = True

    def __call__(self, features: list[dict[str, list[int] | torch.Tensor]]) -> dict[str, torch.Tensor]:
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        label_features = [{"input_ids": feature["labels"]} for feature in features]

        batch = self.processor.pad(input_features, padding=self.padding, return_tensors="pt")
        labels_batch = self.processor.pad(labels=label_features, padding=self.padding, return_tensors="pt")

        # Remplace le remplissage par -100 pour l'ignorer dans la perte.
        batch["labels"] = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        return batch


data_collator = DataCollatorCTCWithPadding(processor=processor)

## 8. Définir la métrique

Le **WER** (*word error rate*) est la proportion de mots mal transcrits : 0 est parfait, et un score plus bas
est meilleur. Le modèle renvoie, pour chaque trame, un score par caractère du vocabulaire. Le caractère le
plus probable de chaque trame est retenu, puis le décodeur CTC fusionne les répétitions et retire les `[PAD]`.

Les références sont décodées avec `group_tokens=False`. Sinon, les lettres doublées seraient fusionnées et
`"hello"` deviendrait `"helo"`.

In [ ]:
import evaluate
import numpy as np

wer_metric = evaluate.load("wer")


def compute_metrics(pred):
    pred_ids = np.argmax(pred.predictions, axis=-1)
    label_ids = np.where(pred.label_ids != -100, pred.label_ids, processor.tokenizer.pad_token_id)

    pred_str = processor.batch_decode(pred_ids)
    label_str = processor.batch_decode(label_ids, group_tokens=False)

    return {"wer": wer_metric.compute(predictions=pred_str, references=label_str)}

## 9. Charger le modèle pré-entraîné

Chargez `facebook/w2v-bert-2.0` avec une nouvelle couche de sortie dont la taille est celle de votre
vocabulaire :

- `pad_token_id` désigne le jeton `[PAD]`, qui sert aussi de *blank* CTC ;
- `add_adapter=True` ajoute un petit adaptateur convolutif au-dessus de l'encodeur ;
- les *dropouts* sont désactivés, comme dans l'article d'origine ;
- `ctc_zero_infinity=True` ignore les rares extraits dont la transcription est trop longue pour l'audio.
  Sans ce réglage, ces extraits rendent la perte infinie.

Le message qui indique que certains poids (`lm_head`, `adapter`) sont nouvellement initialisés est normal :
ce sont les couches que vous allez entraîner.

In [ ]:
from transformers import Wav2Vec2BertForCTC

model = Wav2Vec2BertForCTC.from_pretrained(
    "facebook/w2v-bert-2.0",
    attention_dropout=0.0,
    hidden_dropout=0.0,
    feat_proj_dropout=0.0,
    mask_time_prob=0.0,
    layerdrop=0.0,
    ctc_loss_reduction="mean",
    ctc_zero_infinity=True,
    add_adapter=True,
    pad_token_id=processor.tokenizer.pad_token_id,
    vocab_size=len(processor.tokenizer),
)

## 10. Configurer l'entraînement

Les paramètres principaux sont les suivants :

- **`train_sampling_strategy="group_by_length"`** : forme des lots d'extraits de durées proches. Le
  remplissage est ainsi réduit, et l'entraînement va nettement plus vite ;
- **4 extraits par lot, pendant 10 époques**, avec un taux d'apprentissage de `5e-5` et 500 pas de *warmup* ;
- **`bf16` ou `fp16`** : précision mixte. Le `bf16` est utilisé si le GPU le prend en charge (A100, L4), sinon le
  `fp16` (T4) ;
- **évaluation et sauvegarde tous les 600 pas**. À la fin, le meilleur point de sauvegarde selon le WER est rechargé ;
- **`report_to="trackio"`** : envoie les courbes de perte et de WER à Trackio ;
- **`push_to_hub=True`, `hub_strategy="end"`** : le modèle est publié dans un dépôt privé à la fin.

Ces réglages conviennent pour ce corpus ; adaptez-les si vous changez de langue ou de données. Si vous
obtenez une erreur de mémoire (`CUDA out of memory`), passez `per_device_train_batch_size` à 2 et
`gradient_accumulation_steps` à 2.

In [ ]:
from transformers import TrainingArguments

hub_model_id = f"{hf_user}/{repo_name}"
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()

training_args = TrainingArguments(
    output_dir=output_dir,
    train_sampling_strategy="group_by_length",
    length_column_name="input_length",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=1,
    num_train_epochs=10,
    learning_rate=5e-5,
    warmup_steps=500,
    activation_checkpointing=True,
    bf16=use_bf16,
    fp16=not use_bf16,
    eval_strategy="steps",
    eval_steps=600,
    save_steps=600,
    logging_steps=25,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    report_to="trackio",
    project="burkimbia-w2v-bert-moore",
    run_name="w2v-bert-2.0-moore-10-epochs",
    push_to_hub=True,
    hub_model_id=hub_model_id,
    hub_private_repo=True,
    hub_strategy="end",
)

## 11. Lancer l'entraînement

Créez le `Trainer`. Passez le processeur à `processing_class` : il sera enregistré et publié avec le modèle.

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    data_collator=data_collator,
    processing_class=processor,
    compute_metrics=compute_metrics,
)

Lancez l'entraînement. Il dure environ 2 h à 3 h sur un T4. Le WER doit baisser nettement au cours des
premières évaluations.

Pour suivre les courbes en direct sur une machine locale, ouvrez un terminal à la racine du dépôt et lancez :

```bash
uv run trackio show --project burkimbia-w2v-bert-moore
```

In [ ]:
trainer.train()

## 12. Tester le modèle affiné

Calculez le WER final sur le jeu de validation, puis transcrivez quelques extraits que vous pouvez écouter.

In [ ]:
metrics = trainer.evaluate()
print(f"WER (validation) : {metrics['eval_wer']:.3f}")

In [ ]:
model = trainer.model
model.eval()

for sample in raw_validation.select(range(3)):
    audio = sample["audio"]
    inputs = processor(audio["array"], sampling_rate=audio["sampling_rate"], return_tensors="pt").to(model.device)
    with torch.no_grad():
        logits = model(**inputs).logits
    prediction = processor.batch_decode(torch.argmax(logits, dim=-1))[0]

    print("Référence :", sample["text"])
    print("Modèle    :", prediction)
    print()

ipd.Audio(data=raw_validation[0]["audio"]["array"], rate=16_000)

## 13. Publier le modèle sur le Hub

Publiez le modèle, le processeur et une fiche de modèle générée automatiquement dans le dépôt privé
`<votre compte>/w2v-bert-2.0-moore`. Trackio publie aussi les courbes d'entraînement dans un Space
Hugging Face, accessible depuis l'onglet « Training Metrics » de la page du modèle.

In [ ]:
trainer.push_to_hub()
print(f"Modèle publié : https://huggingface.co/{hub_model_id}")

Pour vérifier la publication, rechargez le modèle depuis le Hub et transcrivez un extrait de validation
avec un `pipeline` de reconnaissance vocale.

In [ ]:
from transformers import pipeline

asr = pipeline("automatic-speech-recognition", model=hub_model_id, device=model.device)
audio = raw_validation[1]["audio"]
print("Référence :", raw_validation[1]["text"])
print("Modèle    :", asr({"raw": audio["array"], "sampling_rate": audio["sampling_rate"]})["text"])